In [23]:
import os
import numpy as np
from glob import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,   Dataset
import  matplotlib.pyplot as plt
import cv2
import torchvision.transforms as T
from scipy.io import loadmat
from torchvision import transforms
from PIL import Image


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNetResNetBackbone(nn.Module):
    def __init__(self):
        super(UNetResNetBackbone, self).__init__()

        # Example Encoder part (adjust according to your backbone)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        # Decoder part (after each upsampling, we apply convolution to match channels)
        self.decoder4 = nn.Conv2d(256, 128, kernel_size=3, padding=1)  # To adjust channel size
        self.decoder3 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.decoder2 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.final_conv = nn.Conv2d(32, 1, kernel_size=1)  # Final output layer

    def forward(self, x):
        # Encoder path
        e1 = self.encoder[0:3](x)
        e2 = self.encoder[3:5](e1)
        e3 = self.encoder[5:](e2)

        # Decoder path
        # Correct channel size adjustments after upsampling
        d4 = F.interpolate(e3, e2.size()[2:], mode='bilinear', align_corners=True)  # Upsample
        d4 = self.decoder4(d4)  # Convolution to adjust channel size
        d3 = F.interpolate(d4, e1.size()[2:], mode='bilinear', align_corners=True)  # Upsample
        d3 = self.decoder3(d3)  # Convolution to adjust channel size
        d2 = F.interpolate(d3, x.size()[2:], mode='bilinear', align_corners=True)  # Upsample
        d2 = self.decoder2(d2)  # Convolution to adjust channel size

        # Output layer
        out = self.final_conv(d2)
        return out


In [25]:
class ShanghaiTechCrowdCountingDataset(Dataset):
    def __init__(self, image_dir, gt_dir, transform=None, img_size=(256, 256)):
        self.image_dir = image_dir
        self.gt_dir = gt_dir
        self.transform = transform
        self.img_size = img_size
        self.image_files = sorted(os.listdir(image_dir))  
        self.gt_files = sorted(os.listdir(gt_dir))  

    def __len__(self):
        return len(self.image_files)

    def generate_density_map(self, img, coordinates):
        density_map = np.zeros(img.shape[:2], dtype=np.float32)

        kernel_size = 15  
        sigma = 4.0  

        for coord in coordinates:
            x, y = int(coord[0]), int(coord[1])
            
            gaussian_kernel = self.create_gaussian_kernel(kernel_size, sigma)
            
            x_min = max(x - kernel_size // 2, 0)
            y_min = max(y - kernel_size // 2, 0)
            x_max = min(x + kernel_size // 2, img.shape[1])
            y_max = min(y + kernel_size // 2, img.shape[0])

            if x_max > x_min and y_max > y_min:
                density_map[y_min:y_max, x_min:x_max] += gaussian_kernel[:y_max - y_min, :x_max - x_min]

        density_map = np.clip(density_map, 0, 255)
        return density_map


    def create_gaussian_kernel(self, kernel_size, sigma):
        ax = np.linspace(-(kernel_size // 2), kernel_size // 2, kernel_size)
        xx, yy = np.meshgrid(ax, ax)
        kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
        kernel /= np.sum(kernel)  
        return kernel

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        gt_path = os.path.join(self.gt_dir, self.gt_files[idx])
        
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  
        
        mat_data = loadmat(gt_path)
        coordinates = mat_data['image_info'][0, 0]['location'][0, 0]  

        density_map = self.generate_density_map(image, coordinates)
        
        image = cv2.resize(image, self.img_size)
        density_map = cv2.resize(density_map, self.img_size)

        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        
        density_map = torch.tensor(density_map, dtype=torch.float32).unsqueeze(0)  

        return image, density_map

In [26]:
image_dir = "ShanghaiTech/part_B_final/train_data/images"
gt_dir = "ShanghaiTech/part_B_final/train_data/ground_truth"

transform = transforms.Compose([
    transforms.Resize((256, 256)),  
    transforms.ToTensor(),  
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])
# transform = transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(30),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
# ])


train_dataset = ShanghaiTechCrowdCountingDataset(image_dir=image_dir, gt_dir=gt_dir, transform=transform)

batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

for images, density_maps in train_loader:
    print("Batch of images shape:", images.shape)  
    print("Batch of density maps shape:", density_maps.shape)  
    break  


Batch of images shape: torch.Size([4, 3, 256, 256])
Batch of density maps shape: torch.Size([4, 1, 256, 256])


In [27]:
image_dir = "ShanghaiTech/part_B_final/test_data/images"
gt_dir = "ShanghaiTech/part_B_final/test_data/ground_truth"
val_dataset = ShanghaiTechCrowdCountingDataset(image_dir=image_dir, gt_dir=gt_dir, transform=transform)

batch_size = 4
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

for images, density_maps in val_loader:
    print("Batch of images shape:", images.shape)  
    print("Batch of density maps shape:", density_maps.shape)  
    break  


Batch of images shape: torch.Size([4, 3, 256, 256])
Batch of density maps shape: torch.Size([4, 1, 256, 256])


In [ ]:
import torch.optim as optim
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model =UNetResNetBackbone()

# Set up optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

model = model.to(device)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for imgs, gt_maps in train_loader:
        # Move data to the correct device
        imgs, gt_maps = imgs.to(device), gt_maps.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(imgs)
        
        # Compute loss
        loss = criterion(outputs, gt_maps)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")


    


Epoch 1/20, Loss: 2.988353072623795e-05
Epoch 2/20, Loss: 2.2564200071428787e-06
Epoch 3/20, Loss: 1.8846629109248169e-06
Epoch 4/20, Loss: 1.6950875857446591e-06
Epoch 5/20, Loss: 1.5712129555822684e-06
Epoch 6/20, Loss: 1.4926155643024685e-06


In [ ]:
import numpy as np
from skimage.measure import label

def count_people(density_map):
    # Count local maxima or threshold the density map
    threshold = 0.05  # Change this value based on your dataset
    density_map[density_map < threshold] = 0
    # Count connected components (i.e., people)
    
    labeled_map = label(density_map > threshold)
    return np.max(labeled_map)

# Validation loop
model.eval()
with torch.no_grad():
    for imgs, gt_maps in val_loader:
        imgs, gt_maps = imgs.to(device), gt_maps.to(device)
        
        # Predict density map
        pred_density_map = model(imgs).cpu().numpy()
        pred_density_map = np.squeeze(pred_density_map)  # Remove unnecessary dimensions
        
        # Count the number of people
        pred_count = count_people(pred_density_map)
        
        print(f"Predicted people count: {pred_count}")


In [ ]:
import matplotlib.pyplot as plt

def plot_results(image, gt_map, pred_map):
    plt.figure(figsize=(12, 6))

    # Original Image
    plt.subplot(1, 3, 1)
    plt.imshow(image.transpose(1, 2, 0))  # Convert from (C, H, W) to (H, W, C)
    plt.title("Original Image")
    plt.axis('off')

    # Ground Truth Density Map
    plt.subplot(1, 3, 2)
    plt.imshow(gt_map.squeeze(), cmap='jet')  # Ensure that gt_map is a 2D array
    plt.title("Ground Truth Density Map")
    plt.axis('off')

    # Predicted Density Map
    plt.subplot(1, 3, 3)
    plt.imshow(pred_map.squeeze(), cmap='jet')  # Ensure that pred_map is a 2D array
    plt.title("Predicted Density Map")
    plt.axis('off')

    # Show the plot
    plt.tight_layout()
    plt.show()

